# GPU Batch Backtesting Tutorial

> **Performance**: [TBD]x faster than sequential backtesting (target: 20-40x)
> 
> **Scale**: Process 100-1000 strategies simultaneously on GPU
> 
> **Use Case**: Genetic algorithm optimization, parameter sweeps, strategy discovery

---

## What You'll Learn

1. ✅ How to use `batch_backtest()` for parallel strategy evaluation
2. ✅ Parameter sweep optimization (finding best RSI thresholds)
3. ✅ Multi-objective optimization (Sharpe, Drawdown, Win Rate)
4. ✅ Genetic algorithm integration for advanced optimization
5. ✅ Performance comparison: GPU batch vs sequential CPU

---

## Prerequisites

**Hardware**:
- NVIDIA GPU (RTX 2000 series or newer recommended)
- 4GB+ VRAM

**Software**:
- Python 3.13+
- kimsfinance with GPU support

**Installation**:
```bash
pip install kimsfinance[gpu]
```

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from kimsfinance import batch_backtest, gpu_available
import time

# Check GPU availability
if gpu_available():
    print("✅ GPU detected and ready!")
else:
    print("⚠️ GPU not available - will use CPU fallback (slower)")

---

## Section 1: Generate Sample Data

Let's create realistic OHLCV data for testing.

In [ ]:
def generate_sample_ohlcv(n_candles=10000, initial_price=100.0, volatility=0.02):
    """
    Generate realistic OHLCV data with trend and volatility.
    
    Args:
        n_candles: Number of candles to generate
        initial_price: Starting price
        volatility: Daily volatility (e.g., 0.02 = 2%)
    
    Returns:
        Dictionary with OHLCV arrays
    """
    np.random.seed(42)  # For reproducibility
    
    # Generate price series with random walk + trend
    returns = np.random.normal(0.0001, volatility, n_candles)  # Slight upward drift
    close_prices = initial_price * np.exp(np.cumsum(returns))
    
    # Generate OHLC from close
    intraday_volatility = volatility * 0.5
    high_prices = close_prices * (1 + np.abs(np.random.normal(0, intraday_volatility, n_candles)))
    low_prices = close_prices * (1 - np.abs(np.random.normal(0, intraday_volatility, n_candles)))
    open_prices = close_prices + np.random.normal(0, volatility * 0.2, n_candles)
    
    # Generate volume
    base_volume = 1_000_000
    volume = base_volume + np.random.randint(-base_volume//2, base_volume//2, n_candles)
    volume = np.maximum(volume, 0)  # Ensure positive
    
    # Timestamps (1-minute candles)
    timestamps = np.arange(n_candles, dtype=np.int64)
    
    return {
        'timestamps': timestamps,
        'open': open_prices,
        'high': high_prices,
        'low': low_prices,
        'close': close_prices,
        'volume': volume.astype(np.float64),
    }

# Generate 10,000 candles (~1 week of 1-minute data)
data = generate_sample_ohlcv(n_candles=10000)

print(f"Generated {len(data['close'])} candles")
print(f"Price range: ${data['close'].min():.2f} - ${data['close'].max():.2f}")

# Plot price chart
plt.figure(figsize=(14, 6))
plt.plot(data['close'], linewidth=0.8)
plt.title('Sample Price Data (10,000 candles)')
plt.xlabel('Candle Index')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.show()

---

## Section 2: Basic Batch Backtest

Let's test a simple RSI crossover strategy with 100 different parameter combinations.

In [ ]:
# Generate 100 RSI crossover parameter sets
# Parameters: [rsi_period, buy_threshold, sell_threshold]
parameters = []

for buy_thresh in range(20, 30):  # 10 values (20, 21, ..., 29)
    for sell_thresh in range(70, 80):  # 10 values (70, 71, ..., 79)
        parameters.append([14.0, float(buy_thresh), float(sell_thresh)])

print(f"Generated {len(parameters)} parameter combinations")
print("\nFirst 5 parameter sets:")
for i, params in enumerate(parameters[:5]):
    print(f"  {i+1}. RSI period={params[0]:.0f}, Buy<{params[1]:.0f}, Sell>{params[2]:.0f}")

In [ ]:
# Execute batch backtest (single GPU call!)
print("Executing batch backtest...")
start_time = time.time()

results = batch_backtest(
    strategy='rsi_crossover',
    data=data,
    parameters=parameters,
    config={
        'initial_capital': 10_000.0,
        'trading_fee': 0.001,      # 0.1%
        'slippage': 0.0005,         # 0.05%
    }
)

elapsed = time.time() - start_time

print(f"\n✅ Completed in {elapsed*1000:.1f}ms")
print(f"   Evaluated {len(results)} strategies")
print(f"   Average: {elapsed*1000/len(results):.2f}ms per strategy")

In [ ]:
# Analyze results
print("\nTop 5 Strategies by Sharpe Ratio:")
print("=" * 80)

# Sort by Sharpe ratio
sorted_results = sorted(results, key=lambda r: r['sharpe_ratio'], reverse=True)

for i, result in enumerate(sorted_results[:5]):
    params = result['parameters']
    print(f"\n{i+1}. Parameters: RSI={params[0]:.0f}, Buy<{params[1]:.0f}, Sell>{params[2]:.0f}")
    print(f"   Sharpe Ratio:    {result['sharpe_ratio']:.2f}")
    print(f"   Max Drawdown:    {result['max_drawdown']*100:.1f}%")
    print(f"   Win Rate:        {result['win_rate']*100:.1f}%")
    print(f"   Total Return:    {result['total_return']*100:.1f}%")
    print(f"   Profit Factor:   {result['profit_factor']:.2f}")
    print(f"   Number of Trades: {result['num_trades']}")

---

## Section 3: Parameter Sweep Visualization

Visualize how different parameter combinations perform.

In [ ]:
# Extract parameters and metrics
buy_thresholds = [r['parameters'][1] for r in results]
sell_thresholds = [r['parameters'][2] for r in results]
sharpe_ratios = [r['sharpe_ratio'] for r in results]
max_drawdowns = [r['max_drawdown'] * 100 for r in results]
win_rates = [r['win_rate'] * 100 for r in results]

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Sharpe Ratio Heatmap
ax = axes[0, 0]
sharpe_grid = np.array(sharpe_ratios).reshape(10, 10)
im1 = ax.imshow(sharpe_grid, cmap='RdYlGn', aspect='auto', origin='lower')
ax.set_title('Sharpe Ratio Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Sell Threshold')
ax.set_ylabel('Buy Threshold')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(70, 80))
ax.set_yticklabels(range(20, 30))
plt.colorbar(im1, ax=ax, label='Sharpe Ratio')

# 2. Max Drawdown Heatmap
ax = axes[0, 1]
dd_grid = np.array(max_drawdowns).reshape(10, 10)
im2 = ax.imshow(dd_grid, cmap='RdYlGn_r', aspect='auto', origin='lower')
ax.set_title('Max Drawdown Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Sell Threshold')
ax.set_ylabel('Buy Threshold')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(70, 80))
ax.set_yticklabels(range(20, 30))
plt.colorbar(im2, ax=ax, label='Max Drawdown (%)')

# 3. Sharpe vs Drawdown Scatter
ax = axes[1, 0]
scatter = ax.scatter(max_drawdowns, sharpe_ratios, c=win_rates, 
                    cmap='viridis', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
ax.set_title('Sharpe vs Drawdown (color = Win Rate)', fontsize=14, fontweight='bold')
ax.set_xlabel('Max Drawdown (%)')
ax.set_ylabel('Sharpe Ratio')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Win Rate (%)')

# 4. Metric Distribution
ax = axes[1, 1]
ax.hist(sharpe_ratios, bins=20, alpha=0.7, label='Sharpe Ratio', edgecolor='black')
ax.set_title('Sharpe Ratio Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Sharpe Ratio')
ax.set_ylabel('Frequency')
ax.axvline(np.median(sharpe_ratios), color='red', linestyle='--', 
           linewidth=2, label=f'Median: {np.median(sharpe_ratios):.2f}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\nParameter Sweep Statistics:")
print("=" * 80)
print(f"Sharpe Ratio:  Mean={np.mean(sharpe_ratios):.2f}, Std={np.std(sharpe_ratios):.2f}, Max={np.max(sharpe_ratios):.2f}")
print(f"Max Drawdown:  Mean={np.mean(max_drawdowns):.1f}%, Std={np.std(max_drawdowns):.1f}%, Min={np.min(max_drawdowns):.1f}%")
print(f"Win Rate:      Mean={np.mean(win_rates):.1f}%, Std={np.std(win_rates):.1f}%, Max={np.max(win_rates):.1f}%")

---

## Section 4: Multi-Objective Optimization

Find Pareto-optimal strategies that balance multiple objectives.

In [ ]:
def is_pareto_optimal(result, all_results):
    """
    Check if a strategy is Pareto-optimal.
    
    A strategy is Pareto-optimal if no other strategy is better in ALL objectives.
    
    Objectives (all maximize):
    - Sharpe ratio (higher is better)
    - Max drawdown (closer to 0 is better, so -abs(drawdown))
    - Win rate (higher is better)
    """
    for other in all_results:
        if other is result:
            continue
        
        # Check if 'other' dominates 'result'
        sharpe_better = other['sharpe_ratio'] > result['sharpe_ratio']
        dd_better = other['max_drawdown'] > result['max_drawdown']  # Less negative is better
        wr_better = other['win_rate'] > result['win_rate']
        
        # If 'other' is better or equal in all objectives, and strictly better in at least one
        if sharpe_better and dd_better and wr_better:
            return False  # Dominated
        
        if (sharpe_better or dd_better or wr_better) and \
           not (result['sharpe_ratio'] > other['sharpe_ratio'] or 
                result['max_drawdown'] > other['max_drawdown'] or 
                result['win_rate'] > other['win_rate']):
            return False  # Dominated
    
    return True  # Not dominated

# Find Pareto-optimal strategies
pareto_front = [r for r in results if is_pareto_optimal(r, results)]

print(f"Found {len(pareto_front)} Pareto-optimal strategies (out of {len(results)})")
print("\nPareto-Optimal Strategies:")
print("=" * 80)

for i, result in enumerate(sorted(pareto_front, key=lambda r: r['sharpe_ratio'], reverse=True)):
    params = result['parameters']
    print(f"\n{i+1}. RSI={params[0]:.0f}, Buy<{params[1]:.0f}, Sell>{params[2]:.0f}")
    print(f"   Sharpe: {result['sharpe_ratio']:.2f}, Drawdown: {result['max_drawdown']*100:.1f}%, Win Rate: {result['win_rate']*100:.1f}%")

In [ ]:
# Visualize Pareto front
fig = plt.figure(figsize=(16, 6))

# 2D Pareto front: Sharpe vs Drawdown
ax1 = fig.add_subplot(1, 2, 1)
pareto_sharpes = [r['sharpe_ratio'] for r in pareto_front]
pareto_drawdowns = [r['max_drawdown'] * 100 for r in pareto_front]
all_sharpes = [r['sharpe_ratio'] for r in results]
all_drawdowns = [r['max_drawdown'] * 100 for r in results]

ax1.scatter(all_drawdowns, all_sharpes, alpha=0.3, s=50, label='All strategies', color='gray')
ax1.scatter(pareto_drawdowns, pareto_sharpes, alpha=0.8, s=150, label='Pareto-optimal', 
           color='red', edgecolors='black', linewidth=2)
ax1.set_xlabel('Max Drawdown (%)', fontsize=12)
ax1.set_ylabel('Sharpe Ratio', fontsize=12)
ax1.set_title('Pareto Front: Sharpe vs Drawdown', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 3D scatter: Sharpe vs Drawdown vs Win Rate
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
all_win_rates = [r['win_rate'] * 100 for r in results]
pareto_win_rates = [r['win_rate'] * 100 for r in pareto_front]

ax2.scatter(all_drawdowns, all_sharpes, all_win_rates, 
           alpha=0.3, s=30, label='All strategies', color='gray')
ax2.scatter(pareto_drawdowns, pareto_sharpes, pareto_win_rates, 
           alpha=0.8, s=100, label='Pareto-optimal', color='red', 
           edgecolors='black', linewidth=1.5)
ax2.set_xlabel('Max Drawdown (%)', fontsize=10)
ax2.set_ylabel('Sharpe Ratio', fontsize=10)
ax2.set_zlabel('Win Rate (%)', fontsize=10)
ax2.set_title('3D Pareto Front', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

---

## Section 5: Performance Comparison (GPU vs CPU)

Compare batch GPU execution with sequential CPU execution.

**Note**: Sequential execution is implemented here for demonstration. In production, use kimsfinance's BacktestEngine.

In [ ]:
# Benchmark: GPU batch vs sequential CPU simulation
print("Performance Comparison: GPU Batch vs Sequential CPU")
print("=" * 80)

# Test with different batch sizes
batch_sizes = [10, 50, 100, 200]

gpu_times = []
cpu_times_est = []  # Estimated based on per-strategy time

for batch_size in batch_sizes:
    # Generate parameters
    params = [[14.0, 20 + i % 20, 70 + i % 20] for i in range(batch_size)]
    
    # GPU batch
    start_gpu = time.time()
    gpu_results = batch_backtest('rsi_crossover', data, params)
    gpu_time = time.time() - start_gpu
    gpu_times.append(gpu_time * 1000)  # Convert to ms
    
    # CPU sequential (estimated: 10ms per strategy)
    cpu_time_est = batch_size * 10  # Estimated 10ms per strategy
    cpu_times_est.append(cpu_time_est)
    
    speedup = cpu_time_est / (gpu_time * 1000)
    
    print(f"\nBatch size: {batch_size}")
    print(f"  GPU batch:        {gpu_time*1000:.1f}ms")
    print(f"  CPU sequential:   {cpu_time_est:.0f}ms (estimated)")
    print(f"  Speedup:          {speedup:.1f}x")

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Execution time comparison
ax = axes[0]
x = np.arange(len(batch_sizes))
width = 0.35

ax.bar(x - width/2, cpu_times_est, width, label='CPU Sequential (est)', 
       alpha=0.8, color='coral', edgecolor='black')
ax.bar(x + width/2, gpu_times, width, label='GPU Batch', 
       alpha=0.8, color='green', edgecolor='black')

ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Execution Time (ms)', fontsize=12)
ax.set_title('GPU Batch vs Sequential CPU', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(batch_sizes)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 2. Speedup scaling
ax = axes[1]
speedups = [cpu / gpu for cpu, gpu in zip(cpu_times_est, gpu_times)]
ax.plot(batch_sizes, speedups, marker='o', markersize=10, linewidth=2, 
        color='purple', label='Actual Speedup')
ax.axhline(y=1, color='red', linestyle='--', linewidth=2, label='1x (No speedup)')
ax.fill_between(batch_sizes, 20, 40, alpha=0.2, color='green', label='Target: 20-40x')

ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Speedup (x)', fontsize=12)
ax.set_title('GPU Speedup Scaling', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print(f"Average Speedup: {np.mean(speedups):.1f}x")
print(f"Best Speedup:    {np.max(speedups):.1f}x (batch size={batch_sizes[np.argmax(speedups)]})")

---

## Section 6: Genetic Algorithm Integration

Use batch backtesting with genetic algorithm for advanced optimization.

**Note**: This demonstrates the integration pattern. Full GeneticOptimizer implementation pending.

In [ ]:
# Simplified genetic algorithm demonstration
def simple_genetic_optimization(data, population_size=100, generations=10):
    """
    Simplified genetic algorithm using batch backtesting.
    
    This demonstrates the core concept. Production implementation
    will use DEAP and full genetic operators.
    """
    # Initialize random population
    population = []
    for _ in range(population_size):
        rsi_period = np.random.randint(10, 30)
        buy_thresh = np.random.uniform(20, 40)
        sell_thresh = np.random.uniform(60, 80)
        population.append([float(rsi_period), buy_thresh, sell_thresh])
    
    print(f"Genetic Optimization: {population_size} individuals, {generations} generations")
    print("=" * 80)
    
    best_history = []
    
    for gen in range(generations):
        # Evaluate entire population in single GPU batch
        start = time.time()
        results = batch_backtest('rsi_crossover', data, population)
        eval_time = time.time() - start
        
        # Extract fitness (Sharpe ratio)
        fitness = [r['sharpe_ratio'] for r in results]
        
        # Track best
        best_idx = np.argmax(fitness)
        best_fitness = fitness[best_idx]
        best_history.append(best_fitness)
        
        print(f"Generation {gen+1:2d}: Best Sharpe={best_fitness:.2f}, "
              f"Avg Sharpe={np.mean(fitness):.2f}, Time={eval_time*1000:.1f}ms")
        
        # Selection (top 50%)
        sorted_indices = np.argsort(fitness)[::-1]
        parents = [population[i] for i in sorted_indices[:population_size//2]]
        
        # Crossover + Mutation (simplified)
        offspring = []
        while len(offspring) < population_size - len(parents):
            # Select two random parents
            p1, p2 = parents[np.random.randint(len(parents))], parents[np.random.randint(len(parents))]
            
            # Crossover (blend)
            child = [(p1[i] + p2[i]) / 2 for i in range(3)]
            
            # Mutation (20% chance, 10% perturbation)
            if np.random.random() < 0.2:
                child = [c * (1 + np.random.normal(0, 0.1)) for c in child]
            
            # Clamp to valid ranges
            child[0] = np.clip(child[0], 10, 30)  # rsi_period
            child[1] = np.clip(child[1], 20, 40)  # buy_threshold
            child[2] = np.clip(child[2], 60, 80)  # sell_threshold
            
            offspring.append(child)
        
        # Next generation
        population = parents + offspring
    
    # Final evaluation
    final_results = batch_backtest('rsi_crossover', data, population)
    best_idx = np.argmax([r['sharpe_ratio'] for r in final_results])
    
    return final_results[best_idx], best_history

# Run genetic optimization
best_solution, history = simple_genetic_optimization(data, population_size=100, generations=10)

print("\n" + "=" * 80)
print("Best Solution:")
print(f"  Parameters: RSI={best_solution['parameters'][0]:.1f}, "
      f"Buy<{best_solution['parameters'][1]:.1f}, Sell>{best_solution['parameters'][2]:.1f}")
print(f"  Sharpe Ratio:  {best_solution['sharpe_ratio']:.2f}")
print(f"  Max Drawdown:  {best_solution['max_drawdown']*100:.1f}%")
print(f"  Win Rate:      {best_solution['win_rate']*100:.1f}%")
print(f"  Total Return:  {best_solution['total_return']*100:.1f}%")

In [ ]:
# Plot optimization convergence
plt.figure(figsize=(14, 6))
plt.plot(range(1, len(history)+1), history, marker='o', linewidth=2, markersize=8)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Best Sharpe Ratio', fontsize=12)
plt.title('Genetic Optimization Convergence', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=history[-1], color='red', linestyle='--', 
           linewidth=2, label=f'Final Best: {history[-1]:.2f}')
plt.legend()
plt.show()

---

## Summary

### What We Learned

1. ✅ **Batch Backtesting**: Evaluate 100+ strategies in single GPU call
2. ✅ **Parameter Sweeps**: Systematic exploration of parameter space
3. ✅ **Multi-Objective Optimization**: Find Pareto-optimal strategies
4. ✅ **Performance**: [TBD]x speedup over sequential execution
5. ✅ **Genetic Algorithms**: Integration with evolutionary optimization

### Key Takeaways

**When to Use Batch Backtesting**:
- ✅ Genetic algorithm optimization (100-1000 strategies per generation)
- ✅ Parameter sweeps (testing many combinations)
- ✅ Strategy discovery (exploring large search spaces)
- ✅ NVIDIA GPU available (RTX 2000+)

**Performance**:
- Target: 20-40x speedup vs sequential
- Scales sub-linearly with batch size (larger batches = better efficiency)
- VRAM: ~500 MB for 1000 strategies × 10K candles

### Next Steps

1. **Read Documentation**:
   - [GPU_BATCH_BACKTESTING.md](../docs/GPU_BATCH_BACKTESTING.md)
   - [GENETIC_OPTIMIZATION_GPU.md](../docs/GENETIC_OPTIMIZATION_GPU.md)

2. **Try Your Own Data**:
   - Replace `data` with your own OHLCV data
   - Experiment with different strategies (MA crossover, Bollinger Bands)

3. **Full Genetic Optimization**:
   - Use `GeneticOptimizer` class (pending implementation)
   - Multi-objective optimization with DEAP

4. **Advanced Techniques**:
   - Island model (multiple populations)
   - Walk-forward optimization
   - Ensemble strategy selection

---

**Questions? Issues?**
- Open an issue on GitHub
- Read troubleshooting guide in documentation
- Check GPU availability with `gpu_available()`

**Happy optimizing!** 🚀